In [ ]:
import csv
import pandas as pd
import re
import os
from bs4 import BeautifulSoup
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
token = os.getenv('HF_TOKEN')
login(token)

News_data = pd.read_csv('guardian_data_since_20250601.csv',encoding='utf-8')
News_data['date'] = pd.to_datetime(News_data['webPublicationDate']).dt.strftime('%y%m%d')
News_data.drop(columns=['webPublicationDate'], inplace=True)

#8sec


In [ ]:
News_data['headline'] = None
News_data['body_text'] = None

for i, item in enumerate(News_data['fields']):
    headline = re.search(r"'headline': '(.*?)', 'body'", item)
    body = re.search(r"'body': '(.*?)'}", item)
    News_data.at[i, 'headline'] = headline.group(1)
    soup = BeautifulSoup(body.group(1))
    body_text = soup.get_text()
    News_data.at[i, 'body_text'] = body_text

#30sec

In [ ]:
prosus_model= AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
prosus_tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')


nlp1 = pipeline('text-classification', model=prosus_model, tokenizer=prosus_tokenizer, truncation=True, max_length=512,batch_size=32)

def sentiment(texts):
    results1 = nlp1([i if isinstance(i, str) and i.strip() else ' ' for i in texts])
    return [ i['score'] if i['label'] == 'positive' else -i['score'] if i['label'] == 'negative' else 0 for i in results1] 

News_data['title_score_prosus'] = sentiment(News_data['headline'])
News_data['content_score_prosus'] = sentiment(News_data['body_text'])
News_data['final_score_prosus'] = News_data[['title_score_prosus','content_score_prosus']].mean(axis=1)

In [ ]:
News_data